# Notebook 3: Cross-Repository Lane Comparison

**Goal:** Compare the general LLM and agentic lanes on shared outcomes.

**Inputs:** `evaluation_attempts.csv`, `evaluation_candidates.csv`, `evaluation_repositories.csv`

**RQs addressed:** RQ4, RQ6, RQ9

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.normalize import build_paired_comparison
from analysis.statistics import (
    mcnemar_test,
    wilcoxon_signed_rank,
    mann_whitney_u,
    bootstrap_ci,
    repository_blocked_bootstrap,
)
from analysis.plots import (
    paired_win_loss_matrix,
    metric_delta_distribution,
    cost_vs_improvement_scatter,
)

sns.set_theme(style='whitegrid')

DATA_DIR   = Path('../output')
attempts   = pd.read_csv(DATA_DIR / 'evaluation_attempts.csv')
candidates = pd.read_csv(DATA_DIR / 'evaluation_candidates.csv')
repos      = pd.read_csv(DATA_DIR / 'evaluation_repositories.csv') if (DATA_DIR / 'evaluation_repositories.csv').exists() else pd.DataFrame()

## 1. Shared Outcome Definitions

Both lanes are compared on:
- `validated_success` — strongest lane-appropriate validation signal
- `coverage_delta` — line coverage change at attempt level
- `mutation_score_delta` — mutation score change at attempt level
- `duration_seconds` — wall-clock runtime

Lane-specific metrics (Roslyn diagnostics, repair counts, run_status) are **not** compared across lanes.

## 2. Candidate-Paired Comparison

In [ ]:
paired = build_paired_comparison(candidates)
print(f'Paired candidate rows: {len(paired)}')
print(paired['winner'].value_counts())

fig, ax = plt.subplots(figsize=(8, 4))
paired_win_loss_matrix(paired, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# McNemar's test on paired candidates where both lanes ran
both = paired[paired['winner'].isin(['llm_won', 'agentic_won', 'tie_success', 'tie_failure'])]
if len(both) >= 5:
    result = mcnemar_test(both['llm_validated_success'], both['agentic_validated_success'])
    print(f"McNemar's: stat={result.statistic:.3f}, p={result.p_value:.4f}, n={result.n}")
    print(f"  {result.note}")
else:
    print('Not enough paired candidates for McNemar test.')

## 3. Repository-Weighted Comparison

In [ ]:
if not repos.empty and 'validated_success_rate' in repos.columns:
    display(repos.groupby('lane')[['validated_success_rate', 'mean_coverage_delta', 'mean_mutation_delta']].describe())
else:
    # Fall back to attempt-level repository-blocked bootstrap
    if 'repository_key' in attempts.columns and 'validated_success' in attempts.columns:
        for lane in ('llm', 'agentic'):
            sub = attempts[attempts['lane'] == lane]
            if len(sub) < 2:
                continue
            ci = repository_blocked_bootstrap(sub, lambda df: df['validated_success'].mean())
            print(f"{lane}: success_rate={ci.estimate:.3f} [{ci.lower:.3f}, {ci.upper:.3f}]")

## 4. Attempt-Weighted Comparison

In [ ]:
if 'lane' in attempts.columns and 'validated_success' in attempts.columns:
    table = attempts.groupby('lane')['validated_success'].agg(['sum', 'count', 'mean'])
    table.columns = ['successes', 'total', 'success_rate']
    display(table)

# Wilcoxon on coverage_delta (attempt-weighted, no pairing)
if 'lane' in attempts.columns and 'coverage_delta' in attempts.columns:
    llm_d = attempts[attempts['lane'] == 'llm']['coverage_delta'].dropna()
    ag_d  = attempts[attempts['lane'] == 'agentic']['coverage_delta'].dropna()
    if len(llm_d) >= 2 and len(ag_d) >= 2:
        result = mann_whitney_u(llm_d, ag_d)
        print(f"Mann-Whitney U (coverage_delta): stat={result.statistic:.1f}, p={result.p_value:.4f}, n={result.n}")

## 5. Cost-Effectiveness

In [ ]:
if all(c in attempts.columns for c in ['duration_seconds', 'coverage_delta', 'lane']):
    fig, ax = plt.subplots(figsize=(8, 5))
    cost_vs_improvement_scatter(attempts, 'duration_seconds', 'coverage_delta', 'lane', ax=ax)
    plt.show()

# Successes per minute
if 'duration_seconds' in attempts.columns and 'validated_success' in attempts.columns:
    spm = attempts.groupby('lane').apply(
        lambda g: g['validated_success'].sum() / (g['duration_seconds'].sum() / 60)
    )
    print('Validated successes per minute:')
    print(spm)

## 6. Risk / Change Footprint

In [ ]:
agentic = attempts[attempts.get('lane', pd.Series()) == 'agentic']
footprint_cols = [c for c in ['changed_files_count', 'production_files_changed',
                               'project_files_changed', 'deleted_files_count'] if c in agentic.columns]
if footprint_cols:
    display(agentic[footprint_cols].describe().T)

# Production/project file change rate
if 'production_files_changed' in agentic.columns:
    rate = (agentic['production_files_changed'].fillna(0) > 0).mean()
    print(f'Agentic production-file change rate: {rate:.1%}')

## 7. Sensitivity Analyses

In [ ]:
# Compare attempt-weighted vs candidate-weighted success rates
print('=== Attempt-weighted ===')
if 'lane' in attempts.columns and 'validated_success' in attempts.columns:
    print(attempts.groupby('lane')['validated_success'].mean())

print('\n=== Candidate-weighted ===')
if 'lane' in candidates.columns and 'any_validated_success' in candidates.columns:
    print(candidates.groupby('lane')['any_validated_success'].mean())